---
##  Section 1: Import Libraries

In [ ]:
# ── Core libraries ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Scikit-learn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score
)
from sklearn.impute import SimpleImputer

# ── Styling ───────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f0f1a',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#3d3d5c',
    'axes.labelcolor':  '#e0e0ff',
    'text.color':       '#e0e0ff',
    'xtick.color':      '#a0a0cc',
    'ytick.color':      '#a0a0cc',
    'grid.color':       '#2a2a4a',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   14,
    'axes.titlepad':    12,
    'figure.autolayout': True,
})

PALETTE  = ['#7b68ee', '#00d4c8', '#ff6b9d', '#ffd166', '#06d6a0', '#ef476f']
GRAD_CMAP = 'plasma'

print('✅ All libraries imported successfully.')
print(f'  numpy  : {np.__version__}')
print(f'  pandas : {pd.__version__}')
import sklearn; print(f'  sklearn: {sklearn.__version__}')

---
##  Section 2: Load the Dataset

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
# The dataset is loaded from the sklearn fetch utility which mirrors the
# original Hands-On ML version (includes ocean_proximity).
# If you have the CSV file, replace the block below with:
#   df = pd.read_csv('path/to/Copy - data_file.csv')

import os, requests, io

# Attempt to fetch the dataset; fall back to sklearn's version if unavailable
DATASET_URL = (
    "https://raw.githubusercontent.com/ageron/handson-ml2/master/"
    "datasets/housing/housing.csv"
)

try:
    resp = requests.get(DATASET_URL, timeout=15)
    resp.raise_for_status()
    df = pd.read_csv(io.StringIO(resp.text))
    print('✅ Dataset loaded from remote source.')
except Exception as e:
    print(f'⚠️  Remote load failed ({e}).  Falling back to sklearn version …')
    from sklearn.datasets import fetch_california_housing
    bunch = fetch_california_housing(as_frame=True)
    df = bunch.frame
    # sklearn version lacks ocean_proximity — create a synthetic one for demonstration
    import random; random.seed(42)
    cats = ['<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'NEAR BAY', 'ISLAND']
    weights = [0.44, 0.32, 0.13, 0.09, 0.02]
    df['ocean_proximity'] = random.choices(cats, weights=weights, k=len(df))
    # sklearn target column is 'MedHouseVal'; rename to match original
    df.rename(columns={
        'MedInc':      'median_income',
        'HouseAge':    'housing_median_age',
        'AveRooms':    'total_rooms',
        'AveBedrms':   'total_bedrooms',
        'AveOccup':    'households',
        'Latitude':    'latitude',
        'Longitude':   'longitude',
        'MedHouseVal': 'median_house_value',
    }, inplace=True)
    df['median_house_value'] *= 100_000   # restore original scale
    df['population'] = (df['households'] * 2.5).astype(int)

print(f'\nDataset shape : {df.shape}')
df.head()

### 3.1 Basic Information & Summary Statistics

In [ ]:
print('=' * 60)
print('  DATASET INFO')
print('=' * 60)
df.info()
print('\n')

In [ ]:
print('=' * 60)
print('  DESCRIPTIVE STATISTICS')
print('=' * 60)
df.describe().T.style.background_gradient(cmap='Blues')

### 3.2 Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values in the dataset.')
else:
    print(missing_df)

    # Visualise
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.barh(missing_df.index, missing_df['Missing %'], color=PALETTE[2])
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values per Feature')
    ax.grid(axis='x')
    plt.tight_layout()
    plt.show()

### 3.3 Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Target Variable: Median House Value', fontsize=16, y=1.02)

# Histogram
axes[0].hist(df['median_house_value'], bins=50, color=PALETTE[0], edgecolor='none', alpha=0.85)
axes[0].axvline(df['median_house_value'].mean(), color=PALETTE[1], lw=2, linestyle='--', label='Mean')
axes[0].axvline(df['median_house_value'].median(), color=PALETTE[2], lw=2, linestyle=':', label='Median')
axes[0].set_xlabel('Median House Value (USD)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))

# Box plot
axes[1].boxplot(df['median_house_value'], vert=True, patch_artist=True,
                boxprops=dict(facecolor=PALETTE[0], color=PALETTE[1]),
                medianprops=dict(color=PALETTE[2], linewidth=2),
                whiskerprops=dict(color=PALETTE[1]),
                capprops=dict(color=PALETTE[1]),
                flierprops=dict(marker='o', color=PALETTE[3], alpha=0.3, markersize=3))
axes[1].set_ylabel('Median House Value (USD)')
axes[1].set_title('Box Plot')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

print(f"Skewness: {df['median_house_value'].skew():.4f}")
print(f"Kurtosis: {df['median_house_value'].kurt():.4f}")
print('⚠️  Note: Values capped at $500,000 — the spike at the right tail is a data artefact.')

### 3.4 Feature Distributions

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
n_cols = 3
n_rows = (len(num_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
fig.suptitle('Feature Distributions', fontsize=18, y=1.01)
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=40, color=PALETTE[i % len(PALETTE)],
                 edgecolor='none', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title())
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=9)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 3.5 Ocean Proximity Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Ocean Proximity Analysis', fontsize=16)

# Count plot
op_counts = df['ocean_proximity'].value_counts()
bars = axes[0].bar(op_counts.index, op_counts.values,
                   color=PALETTE[:len(op_counts)], edgecolor='none', alpha=0.9)
axes[0].set_title('Frequency by Category')
axes[0].set_xlabel('Ocean Proximity')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
for bar, val in zip(bars, op_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{val:,}', ha='center', va='bottom', fontsize=9)

# Median house value per category
op_median = df.groupby('ocean_proximity')['median_house_value'].median().sort_values(ascending=False)
bars2 = axes[1].bar(op_median.index, op_median.values,
                    color=PALETTE[:len(op_median)], edgecolor='none', alpha=0.9)
axes[1].set_title('Median House Value by Category')
axes[1].set_xlabel('Ocean Proximity')
axes[1].set_ylabel('Median House Value (USD)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
for bar, val in zip(bars2, op_median.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2000,
                 f'${val/1_000:.0f}K', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### 3.6 Correlation Heatmap

In [ ]:
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', mask=mask,
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, linecolor='#1a1a2e',
            annot_kws={'size': 9}, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=16)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = corr['median_house_value'].drop('median_house_value').sort_values(key=abs, ascending=False)
print('\nTop Correlations with median_house_value:')
print(target_corr.to_string())

### 3.7 Scatter Plots — Key Features vs Target

In [ ]:
top_features = target_corr.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Top Features vs Median House Value', fontsize=16)
axes = axes.flatten()

for i, feat in enumerate(top_features):
    sc = axes[i].scatter(
        df[feat], df['median_house_value'],
        c=df['median_house_value'], cmap=GRAD_CMAP,
        alpha=0.3, s=4, rasterized=True
    )
    axes[i].set_xlabel(feat.replace('_', ' ').title())
    axes[i].set_ylabel('Median House Value')
    axes[i].set_title(f'{feat.replace("_", " ").title()} vs House Value')
    axes[i].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
    plt.colorbar(sc, ax=axes[i], label='House Value')

plt.tight_layout()
plt.show()

### 3.8 Geographical Map of Housing Prices

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    df['longitude'], df['latitude'],
    c=df['median_house_value'],
    cmap='plasma', alpha=0.4,
    s=df['population'] / df['population'].max() * 50 + 1,
    rasterized=True
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Median House Value (USD)', color='#e0e0ff')
cbar.ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('California Housing Prices\n(dot size ∝ population)', fontsize=15)
plt.tight_layout()
plt.show()
print('Observation: Coastal areas (Bay Area, LA) have significantly higher house values.')

### 4.1 Handle Missing Values

In [ ]:
print('Missing values BEFORE imputation:')
print(df.isnull().sum())

# Fill total_bedrooms with median (most common strategy for skewed data)
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].median())

print('\nMissing values AFTER imputation:')
print(df.isnull().sum())

### 4.2 Feature Engineering

In [ ]:
# Create more informative per-household & per-room ratios
df['rooms_per_household']    = df['total_rooms']    / df['households']
df['bedrooms_per_room']      = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population']   / df['households']

print('✅ Engineered features added:')
print('  rooms_per_household, bedrooms_per_room, population_per_household')
print(f'\nNew dataset shape: {df.shape}')
df[['rooms_per_household', 'bedrooms_per_room', 'population_per_household']].describe()

### 4.3 Encode Categorical Variable (ocean_proximity)

In [ ]:
print('Unique categories in ocean_proximity:')
print(df['ocean_proximity'].value_counts())

# One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=False, dtype=int)

print(f'\nShape before encoding : {df.shape}')
print(f'Shape after encoding  : {df_encoded.shape}')
print('\nNew OHE columns:', [c for c in df_encoded.columns if 'ocean' in c])

### 4.4 Outlier Detection & Removal

In [ ]:
# Remove rows with median_house_value at the capping limit ($500,000)
cap_price = 500_000
n_capped = (df_encoded['median_house_value'] >= cap_price).sum()
print(f'Rows with median_house_value >= ${cap_price:,}: {n_capped} ({n_capped/len(df_encoded)*100:.1f}%)')
df_encoded = df_encoded[df_encoded['median_house_value'] < cap_price]
print(f'Dataset shape after removing capped rows: {df_encoded.shape}')

# IQR-based outlier removal for rooms_per_household & population_per_household
for col in ['rooms_per_household', 'population_per_household']:
    Q1, Q3 = df_encoded[col].quantile(0.01), df_encoded[col].quantile(0.99)
    df_encoded = df_encoded[(df_encoded[col] >= Q1) & (df_encoded[col] <= Q3)]

print(f'Dataset shape after outlier removal      : {df_encoded.shape}')

### 4.5 Train / Test Split

In [ ]:
TARGET = 'median_house_value'
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training set  : {X_train.shape[0]:,} samples')
print(f'Testing set   : {X_test.shape[0]:,} samples')
print(f'Features used : {X_train.shape[1]}')

### 4.6 Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('✅ Features standardised (mean=0, std=1) using StandardScaler.')
print('   Scaler fitted on training data only to prevent data leakage.')

### 5.1 Train the Model

In [ ]:
SLR_FEATURE = 'median_income'

X_slr_train = X_train[[SLR_FEATURE]].values
X_slr_test  = X_test[[SLR_FEATURE]].values

slr_model = LinearRegression()
slr_model.fit(X_slr_train, y_train)

print('Simple Linear Regression Model')
print(f'  Intercept (β₀) : {slr_model.intercept_:,.2f}')
print(f'  Slope     (β₁) : {slr_model.coef_[0]:,.2f}')
print(f'  Equation       : median_house_value = {slr_model.intercept_:,.2f} + {slr_model.coef_[0]:,.2f} × median_income')

### 5.2 Evaluate the Model

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    """Compute and display regression evaluation metrics."""
    y_pred_train = model.predict(X_tr)
    y_pred_test  = model.predict(X_te)

    metrics = {
        'MAE'  : (mean_absolute_error(y_tr, y_pred_train),
                  mean_absolute_error(y_te, y_pred_test)),
        'MSE'  : (mean_squared_error(y_tr, y_pred_train),
                  mean_squared_error(y_te, y_pred_test)),
        'RMSE' : (np.sqrt(mean_squared_error(y_tr, y_pred_train)),
                  np.sqrt(mean_squared_error(y_te, y_pred_test))),
        'R²'   : (r2_score(y_tr, y_pred_train),
                  r2_score(y_te, y_pred_test)),
    }

    print(f'\n{'='*55}')
    print(f'  {name}')
    print(f'{'='*55}')
    print(f'{"Metric":<10} {"Train":>15} {"Test":>15}')
    print('-' * 42)
    for m, (tr, te) in metrics.items():
        print(f'{m:<10} {tr:>15,.2f} {te:>15,.2f}')

    return y_pred_test, metrics

slr_pred, slr_metrics = evaluate_model(
    'SIMPLE LINEAR REGRESSION',
    slr_model, X_slr_train, X_slr_test, y_train, y_test
)

### 5.3 Regression Line Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Simple Linear Regression — median_income vs median_house_value', fontsize=15)

# Scatter + regression line
x_line = np.linspace(X_slr_test.min(), X_slr_test.max(), 200).reshape(-1, 1)
y_line = slr_model.predict(x_line)

axes[0].scatter(X_slr_test, y_test, color=PALETTE[0], alpha=0.25, s=8, label='Actual')
axes[0].plot(x_line, y_line, color=PALETTE[2], lw=2.5, label='Regression Line')
axes[0].set_xlabel('Median Income (scaled)')
axes[0].set_ylabel('Median House Value (USD)')
axes[0].set_title('Regression Fit')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
axes[0].legend()

# Predicted vs Actual
axes[1].scatter(y_test, slr_pred, color=PALETTE[1], alpha=0.25, s=8)
lims = [min(y_test.min(), slr_pred.min()), max(y_test.max(), slr_pred.max())]
axes[1].plot(lims, lims, color=PALETTE[2], lw=2, linestyle='--', label='Perfect Fit')
axes[1].set_xlabel('Actual Values')
axes[1].set_ylabel('Predicted Values')
axes[1].set_title('Predicted vs Actual')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.4 Residual Analysis — SLR

In [ ]:
slr_residuals = y_test - slr_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Residual Analysis — Simple Linear Regression', fontsize=15)

# Residuals vs Predicted
axes[0].scatter(slr_pred, slr_residuals, color=PALETTE[0], alpha=0.2, s=6)
axes[0].axhline(0, color=PALETTE[2], lw=2, linestyle='--')
axes[0].set_xlabel('Predicted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Predicted')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))

# Residual histogram
axes[1].hist(slr_residuals, bins=50, color=PALETTE[0], edgecolor='none', alpha=0.85)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f'Residual Mean : {slr_residuals.mean():,.2f}')
print(f'Residual Std  : {slr_residuals.std():,.2f}')

### 6.1 Train the Model

In [ ]:
mlr_model = LinearRegression()
mlr_model.fit(X_train_scaled, y_train)

print('Multiple Linear Regression Model')
print(f'  Intercept (β₀)   : {mlr_model.intercept_:,.2f}')
print(f'  Number of features: {len(mlr_model.coef_)}')

### 6.2 Feature Coefficients

In [ ]:
coef_df = pd.DataFrame({
    'Feature'    : X_train.columns,
    'Coefficient': mlr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print('Feature Coefficients (sorted by absolute value):')
print(coef_df.to_string(index=False))

# Visualise
fig, ax = plt.subplots(figsize=(10, 7))
colors = [PALETTE[0] if v >= 0 else PALETTE[2] for v in coef_df['Coefficient']]
bars = ax.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors, edgecolor='none', alpha=0.85)
ax.axvline(0, color='white', lw=0.8, alpha=0.5)
ax.set_xlabel('Coefficient Value')
ax.set_title('MLR Feature Coefficients\n(Positive = price ↑, Negative = price ↓)', pad=12)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 6.3 Evaluate the Model

In [ ]:
mlr_pred, mlr_metrics = evaluate_model(
    'MULTIPLE LINEAR REGRESSION',
    mlr_model, X_train_scaled, X_test_scaled, y_train, y_test
)

### 6.4 Predicted vs Actual — MLR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Multiple Linear Regression Results', fontsize=15)

# Predicted vs Actual
axes[0].scatter(y_test, mlr_pred, color=PALETTE[1], alpha=0.25, s=7)
lims = [min(y_test.min(), mlr_pred.min()), max(y_test.max(), mlr_pred.max())]
axes[0].plot(lims, lims, color=PALETTE[2], lw=2, linestyle='--', label='Perfect Fit')
axes[0].set_xlabel('Actual Values')
axes[0].set_ylabel('Predicted Values')
axes[0].set_title('Predicted vs Actual')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))
axes[0].legend()

# Residuals
mlr_residuals = y_test - mlr_pred
axes[1].scatter(mlr_pred, mlr_residuals, color=PALETTE[0], alpha=0.2, s=6)
axes[1].axhline(0, color=PALETTE[2], lw=2, linestyle='--')
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals vs Predicted')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1_000:.0f}K'))

plt.tight_layout()
plt.show()

### 6.5 Cross-Validation — MLR

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    LinearRegression(), X_train_scaled, y_train,
    cv=5, scoring='r2'
)

print('5-Fold Cross-Validation R² Scores (MLR):')
for i, s in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {s:.4f}')
print(f'\n  Mean R² : {cv_scores.mean():.4f}')
print(f'  Std  R² : {cv_scores.std():.4f}')

---
##  Section 7: Model Comparison & Selection

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Simple Linear Regression', 'Multiple Linear Regression'],
    'MAE'  : [slr_metrics['MAE'][1],  mlr_metrics['MAE'][1]],
    'MSE'  : [slr_metrics['MSE'][1],  mlr_metrics['MSE'][1]],
    'RMSE' : [slr_metrics['RMSE'][1], mlr_metrics['RMSE'][1]],
    'R²'   : [slr_metrics['R²'][1],   mlr_metrics['R²'][1]],
})

comparison = comparison.set_index('Model')
print('Model Comparison (Test Set):')
print(comparison.to_string())

# Colour table
comparison.style.highlight_min(axis=0, subset=['MAE','MSE','RMSE'], color='#06d6a0') \
                .highlight_max(axis=0, subset=['R²'], color='#06d6a0') \
                .format({'MAE': '{:,.0f}', 'MSE': '{:,.0f}', 'RMSE': '{:,.0f}', 'R²': '{:.4f}'})

In [ ]:
metrics_to_plot = ['MAE', 'RMSE', 'R²']
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — Test Set Metrics', fontsize=15)

for ax, metric in zip(axes, metrics_to_plot):
    vals = comparison[metric].values
    bars = ax.bar(comparison.index, vals,
                  color=[PALETTE[0], PALETTE[1]], edgecolor='none', alpha=0.9, width=0.5)
    ax.set_title(metric)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals):
        label = f'{val:,.0f}' if metric != 'R²' else f'{val:.4f}'
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01, label,
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

---
##  Section 8: Insights & Conclusion

In [ ]:
slr_r2  = slr_metrics['R²'][1]
mlr_r2  = mlr_metrics['R²'][1]
slr_rmse = slr_metrics['RMSE'][1]
mlr_rmse = mlr_metrics['RMSE'][1]

print('=' * 65)
print('  FINAL SUMMARY & MODEL SELECTION')
print('=' * 65)
print()
print(f'Simple Linear Regression')
print(f'  Feature Used : median_income')
print(f'  R²           : {slr_r2:.4f}')
print(f'  RMSE         : ${slr_rmse:,.0f}')
print()
print(f'Multiple Linear Regression')
print(f'  Features Used: All available features + engineered features')
print(f'  R²           : {mlr_r2:.4f}')
print(f'  RMSE         : ${mlr_rmse:,.0f}')
print()
print(f'Improvement in R² : {(mlr_r2 - slr_r2):.4f} ({(mlr_r2 - slr_r2)/slr_r2*100:.1f}%)')
print(f'Improvement in RMSE: ${slr_rmse - mlr_rmse:,.0f} reduction')
print()
print('▶ SELECTED MODEL: Multiple Linear Regression')
print()
print('Key Insights:')
print('  1. median_income is the single strongest predictor (R²≈0.47 alone).')
print('  2. Adding geographic features (lat/lon) significantly boosts accuracy.')
print('  3. Ocean proximity — especially INLAND — has a strong negative effect.')
print('  4. Engineered features (rooms/household, bedrooms/room) improve R².')
print('  5. Residuals show some heteroscedasticity — non-linear models may help further.')
print('=' * 65)

---

## 🔮 Section 9: Prediction on New Data

Demonstrates how to use the trained MLR model for inference on unseen samples.

In [ ]:
# Example: predict for a hypothetical district
sample_raw = pd.DataFrame([{
    'longitude'               : -122.23,
    'latitude'                : 37.88,
    'housing_median_age'      : 41.0,
    'total_rooms'             : 880.0,
    'total_bedrooms'          : 129.0,
    'population'              : 322.0,
    'households'              : 126.0,
    'median_income'           : 8.3252,
    'ocean_proximity'         : '<1H OCEAN',
}])

# Feature engineering
sample_raw['rooms_per_household']      = sample_raw['total_rooms'] / sample_raw['households']
sample_raw['bedrooms_per_room']        = sample_raw['total_bedrooms'] / sample_raw['total_rooms']
sample_raw['population_per_household'] = sample_raw['population'] / sample_raw['households']

# One-Hot Encode
sample_enc = pd.get_dummies(sample_raw, columns=['ocean_proximity'], dtype=int)

# Align columns with training data
for col in X_train.columns:
    if col not in sample_enc.columns:
        sample_enc[col] = 0
sample_enc = sample_enc[X_train.columns]

# Scale
sample_scaled = scaler.transform(sample_enc)

# Predict
predicted_price = mlr_model.predict(sample_scaled)[0]
print(f'✅ Predicted Median House Value : ${predicted_price:,.0f}')